In [113]:
from utils import getDevice, collate_fn #important to impot at the start for reproducibility :3 

In [114]:
# GENERAL IMPORTS
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import time
import torch
import gc
import os
import pandas as pd
from IPython.display import clear_output

from torchvision.models import vgg16
from torchvision import models
from torchvision import transforms
from torchvision.ops import roi_pool
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

# CUSTIM FUNCTIONS AND VARIABLES
from kitty import depth_read, listPicsWith, togglePath, MatchDepthToCar, KITTY_PATH
from yolo import getEmbedFromResults, getCropsFromResults, getCropsFromResult, getEmbedFromCrops, CLASSES_YOLO, CONFIDENCE_YOLO

# YOLO STUFF
from ultralytics import YOLO

#DINO STUFF
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

In [115]:
DEVICE = getDevice()
HEIGHT = 376
WIDTH = 1242
TARGET_TYPES = ['Car']
BATCH_SIZE = 4
NUM_WORKERS = 4

In [116]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((HEIGHT, WIDTH))
])

#DATASET IS LOADED AND DATALOADER IS CREATED

kitty_dataset =torchvision.datasets.Kitti(root="../datasets/", train=True,transform=transform, download=True)
kittty_dataloader = torch.utils.data.DataLoader(
    kitty_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

KITTY_LENGTH = len(kitty_dataset)
BATCH_COUNT = KITTY_LENGTH // BATCH_SIZE + (KITTY_LENGTH % BATCH_SIZE > 0)
print(f"KITTY dataset is length: {KITTY_LENGTH}\nDataloader has length: {len(kittty_dataloader)}\nBatch count {BATCH_COUNT}")

#I try a single batch to analyze the data structure and types
images, targets_batch = next(iter(kittty_dataloader))
print(f"Type of the images data is: {type(images)} type of image {type(images[0])} len: {len(images)} and type of the target is {type(targets_batch)} of len {len(targets_batch)}")

KITTY dataset is length: 7481
Dataloader has length: 1871
Batch count 1871
[2026-04-18 21:32:15.154] [warning] [sycl_collector.h:388] Another subscriber already subscribed to Sycl runtime events, so PTI will not subscribe to them. It will affect correctness of PTI profile: e.g. report zero XPU time for CPU callers of GPU kernels.
Using device: xpu :3
Setting seed to 42:3
[2026-04-18 21:32:16.359] [warning] [sycl_collector.h:388] Another subscriber already subscribed to Sycl runtime events, so PTI will not subscribe to them. It will affect correctness of PTI profile: e.g. report zero XPU time for CPU callers of GPU kernels.
Using device: xpu :3
Setting seed to 42:3
[2026-04-18 21:32:17.586] [warning] [sycl_collector.h:388] Another subscriber already subscribed to Sycl runtime events, so PTI will not subscribe to them. It will affect correctness of PTI profile: e.g. report zero XPU time for CPU callers of GPU kernels.
Using device: xpu :3
Setting seed to 42:3
[2026-04-18 21:32:18.774] [w

In [117]:
for i, targets in enumerate(targets_batch):
    print(f"Target type is {type(targets)} and length is {len(targets)}")
    for target in targets:
        print(type(target))  # Should be a dict
        obj_types  = target["type"]        # e.g. ['Car', 'Pedestrian']
        locations  = target["location"]    # Tensor [N, 3]: (X, Y, Z) in metres
        bboxes     = torch.Tensor(target["bbox"])        # Tensor [N, 4]: (x1, y1, x2, y2) pixels
        print(f"Object types: {obj_types} of type {type(obj_types)}")
        print(f"Locations: {locations} of type {type(locations)}")
        print(f"Bounding boxes: {bboxes} of type {type(bboxes)}")

    print("")

Target type is <class 'list'> and length is 4
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-1.83, 1.5, 69.06] of type <class 'list'>
Bounding boxes: tensor([582.4900, 175.1300, 599.7400, 188.7900]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-7.81, 1.8, 21.31] of type <class 'list'>
Bounding boxes: tensor([286.5300, 180.4200, 396.6000, 239.9200]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-7.69, 1.64, 55.19] of type <class 'list'>
Bounding boxes: tensor([495.7800, 174.6000, 522.8600, 194.8400]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: DontCare of type <class 'str'>
Locations: [-1000.0, -1000.0, -1000.0] of type <class 'list'>
Bounding boxes: tensor([533.5200, 155.2300, 578.3500, 193.8100]) of type <class 'torch.Tensor'>

Target type is <class 'list'> and length is 8
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-3.73,

In [118]:
yolo_embeds_paths =[]


In [ ]:


def extract_crops_depth(images_batch, targets_batch):
    # 1 IMAGES
        stacked_images = torch.stack(images_batch)
        # 2,3 BOXES and DEPTHS
        boxes =[]
        depths = []
        skip = []
        image_indicies = set(range(len(images_batch)))
        keep=[]
        crops=[]
        for i, targets in enumerate(targets_batch):
            boxes_with_target = [torch.tensor(target["bbox"], dtype=torch.int32) for target in targets if target["type"] in TARGET_TYPES]
            depths_with_target = [target["location"][2] for target in targets if target["type"] in TARGET_TYPES]
            if len(boxes_with_target) > 0:
                boxes.append(torch.stack(boxes_with_target))
                depths.append(torch.Tensor(depths_with_target))
            else:
                skip.append(i)

        # 3.5 check that there is actually data 
        if len(boxes) == 0:
            print("No boxes found in this batch, skipping...")
            del boxes, images_batch, targets_batch
            gc.collect()
            return -1, -1

        #3.6 skip images that we do not want and add the depths
        keep = list(image_indicies - set(skip))
        stacked_images = stacked_images[keep,:,:,:]
        depths = torch.cat(depths).cpu()
        #depths_true_all.extend(depths.tolist())

        # 4 crop the images to a specific size
        for i in range(stacked_images.shape[0]):
            # the current data is fetched
            image = stacked_images[i]
            boxes_cur = boxes[i]

            #images are copped to the box
            for box in boxes_cur:
                x1,y1,x2,y2 = box.tolist()
                crop_tensor = image[:, y1:y2, x1:x2]
                crop_np = crop_tensor.permute(1, 2, 0).numpy()
                crop_np = (crop_np * 255).clip(0, 255).astype(np.uint8)
                crops.append(crop_np)

        #exit function and clear the local variables
        del stacked_images, boxes, targets_batch
        return crops, depths.tolist()

def extract_crops(loader: DataLoader):
    """
    Extract the crops before the embeddings are retrieved.

    Args:
        loader: dataloader for the dataset
    """
    depths_true_all = []
    counter = 0
    chunk=[]
    chunk_count=0
    
    for images_batch, targets_batch in loader:
        #preporcess the data, format it in a way that is expected by the model===================
        clear_output(wait=True)
        print(f"Batch {counter} in progress; ({counter/len(loader)*100:.2f}%);")

        #extract boxes and depths to be added and saved==========================================
        crops, depths = extract_crops_depth(images_batch, targets_batch)
        if crops==-1 and depths==-1:
            continue
        depths_true_all.extend(depths)

        #YOLO embeds=============================================================================
        embeds = []
        for crop in crops:
            em = getEmbedFromCrops(crop)
            chunk.append(em[0])

        #flush the batch into the memory
        chumky_path = f"../data/yolo_embeds_chunk_{chunk_count}.pt"
        torch.save(torch.stack(chunk), chumky_path)
        #save the path
        yolo_embeds_paths.append(chumky_path)        

        #clear the variables and incremenet the counter===========================================
        del crops, depths, embeds,chunk
        chunk = []
        chunk_count += 1
        counter+=1

    gc.collect() 
    return depths_true_all

In [120]:
depths_true = extract_crops(kittty_dataloader)

Batch 1870 in progress; (99.95%);
No boxes found in this batch, skipping...


In [121]:
print("Merging YOLO embedding chunks...")
all_chunks = [torch.load(p) for p in yolo_embeds_paths]
full_embeds = torch.cat(all_chunks, dim=0)
torch.save(full_embeds, "../data/embeds_yolo_og_boxes.pt")
print(f"Saved YOLO embeddings: {full_embeds.shape}")
del all_chunks, full_embeds
gc.collect()

for path in yolo_embeds_paths:
    os.remove(path)

Merging YOLO embedding chunks...
Saved YOLO embeddings: torch.Size([28742, 256])


In [123]:
depths_tensor = torch.Tensor(depths_true)
torch.save(depths_tensor, "../data/true_depth.pt")
print(f"Saved the true depths into a file with shape: {depths_tensor.shape}")
del depths_tensor
gc.collect()

Saved the true depths into a file with shape: torch.Size([28742])


666